# Level Configurations

| level   | #scenarios | number of agents           | max. number of intermediate stops | properties                                                                                       | malfunctions                                    |
|---------|:----------:|----------------------------|:---------------------------------:|--------------------------------------------------------------------------------------------------|-------------------------------------------------|
| level_0 |     5      | {8,11,14,26,28}​            | {3,3,4,6,6}                       | One train per Line starting at t=0                                                               | None                                            |
| level_1 |     5      | {36,50,62,118,210}         | {3,3,4,6,6}                       | Multiple trains per Line, different starting times, larger travel factor (more time for journey) | None                                            |
| level_2 |     5      | {90,125,150,300,532}​       | {3,3,4,6,6}                       | More trains, tighter schedules (periodicity & travel factor)                                     | None                                            |
| level_3 |     5      | {36,50,62,118,210}         | {3,3,4,6,6}                       | Like level 1 but with                                                                            | Breakdowns                                      |
| level_4 |     5      | {90,125,150,300,532}​       | {3,3,4,6,6}                       | Like level 2 but with                                                                            | Breakdowns and departure delays                 |
| level_5 |     5      | {90,125,150,300,532}​       | {3,3,4,6,6}                       | Like level 4 but with more severe malfunctions (more frequent & longer)​                          | Breakdowns and departure delays                 |
| level_6 |     5      | {532,532,532,532,532}​      | {6,6,6,6,6}                       | Full map only, progressively more malfunctions (including infrastructure disruptions)            | Breakdowns, departure delays and infrastructure |

## Loading competition environments from ecml2026-starterkit

In [ ]:
from flatland.envs.persistence import RailEnvPersister
from flatland.utils.rendertools import RenderTool
import PIL

In [ ]:
!git clone https://github.com/flatland-association/ecml2026-starterkit -b use-competition-infrastructure-for-top-level-readme

In [ ]:
import sys

sys.path.insert(0, "ecml2026-starterkit")

In [ ]:
!unzip ecml2026-starterkit/reinforcement_learning/curriculum/example_curriculum.zip -d ecml2026-starterkit/reinforcement_learning/curriculum/

In [ ]:
env, _ = RailEnvPersister.load_new("ecml2026-starterkit/reinforcement_learning/curriculum/00_scene_1_ll-2_a-1.pkl")

In [ ]:
env_renderer = RenderTool(env)
image = env_renderer.render_env(show=False, show_observations=False, show_predictions=False, return_image=True)
display(PIL.Image.fromarray(image))

## Deep-Dive Line Generation

In [ ]:
agent = env.agents[0]
agent

Lines consist of a sequence of "flexible waypoints" (a set of routing alternatives). The first waypoint is always unique and the last is a waypoint whose direction is `None` (meaning the cell needs to be reached from any direction). Here's an example of such a simple line:

In [ ]:
agent.waypoints

However, lines can consist of more than source and target....

### Deep-Dive Schedules

The timetable consist of time windows for each flexible waypoint, an earliest arrival and a latest departure . For the source, latest arrival is undefined, for the final target, earliest departure is undefined, obviously:

In [ ]:
agent.waypoints_earliest_departure

agent.waypoints_latest_arrival

The environment does not keep track of intermediate waypoints. The environment only enforces that an agent cannot enter Flatland before earliest departure defined for the agent's initial waypoint. However, the rewards function evaluates whether trains have passed in the requested time windows and will penalize otherwise.